# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_01 — Almgren-Chriss Optimal Execution

### Research question

How do we optimally execute a large order over time when trading too quickly creates market impact, but trading too slowly creates price-risk exposure?

This notebook opens **Phase 6: Execution, Market Microstructure & Systems**.

Phase 5 built the research-hygiene layer: backtests, validation, costs, overfitting controls, live paper monitoring, and tearsheets. Phase 6 moves closer to the trading desk:

```text
06_01_almgren_chriss_optimal_execution
06_02_limit_order_book_simulator
06_03_market_impact_model_calibration
06_04_execution_algorithms_twap_vwap_pov
06_05_order_management_system_simulator
06_06_smart_order_router_simulation
06_07_latency_and_queue_position_model
06_08_microprice_and_short_horizon_alpha
06_09_execution_risk_controls_and_kill_switch
06_10_futures_roll_execution_engine
06_11_l1_bid_ask_backtest_execution_model
06_12_production_reconciliation_and_breaks
06_13_execution_research_capstone
```

This notebook covers:

1. optimal execution problem;
2. implementation shortfall;
3. temporary impact;
4. permanent impact;
5. price-risk term;
6. Almgren-Chriss objective;
7. risk-neutral schedule;
8. risk-averse schedule;
9. closed-form discrete liquidation path;
10. execution half-life;
11. efficient frontier of expected cost versus risk;
12. TWAP comparison;
13. VWAP-style comparison;
14. front-loaded and back-loaded schedules;
15. Monte Carlo implementation shortfall simulation;
16. participation and volume constraints;
17. sensitivity to risk aversion;
18. sensitivity to volatility and impact;
19. calibration notes;
20. governance flags;
21. saved outputs and manifest.

The key idea:

> Optimal execution is a trade-off between market impact cost and the risk of waiting.

## 1. Why optimal execution matters

Suppose a portfolio manager wants to sell $X$ shares.

If they sell immediately:

- market-risk exposure disappears;
- but temporary impact is high.

If they sell slowly:

- temporary impact is lower;
- but the remaining inventory is exposed to price moves.

The execution problem is:

$$
\min_{\{n_k\}} E[C] + \lambda Var(C)
$$

where:

- $n_k$ is the number of shares traded in interval $k$;
- $C$ is implementation shortfall;
- $\lambda$ controls risk aversion.

This is the classic Almgren-Chriss setup.

## 2. Implementation shortfall

For a sell programme, implementation shortfall is approximately:

$$
C = X S_0 - \sum_{k=1}^{N} n_k \tilde S_k
$$

where:

- $X$ is initial inventory;
- $S_0$ is decision price;
- $n_k$ is shares sold at time $k$;
- $\tilde S_k$ is execution price.

If execution prices are worse than the decision price, shortfall is positive.

Execution research often decomposes shortfall into:

$$
\begin{aligned}
Shortfall &= Spread \\
&\quad + TemporaryImpact \\
&\quad + PermanentImpact \\
&\quad + TimingRisk \\
&\quad + Fees
\end{aligned}
$$

## 3. Almgren-Chriss model

A simple discrete-time price process:

$$
S_k = S_{k-1} + \sigma \sqrt{\tau}\epsilon_k - \gamma n_k
$$

where:

- $\sigma$ is volatility per square-root time;
- $\tau$ is interval length;
- $\epsilon_k \sim N(0,1)$;
- $\gamma$ is permanent impact per share.

Execution price for a sell order:

$$
\tilde S_k = S_{k-1} - \eta \frac{n_k}{\tau}
$$

where:

- $\eta$ is temporary impact coefficient;
- $n_k/\tau$ is trading rate.

Temporary impact is paid only while trading.

Permanent impact changes the future price level.

## 4. Cost-risk objective

Let $x_k$ be shares remaining after interval $k$:

$$
x_0=X,\quad x_N=0
$$

$$
n_k = x_{k-1}-x_k
$$

A common Almgren-Chriss objective can be written:

$$
\min_x \sum_{k=1}^{N} \eta \frac{(x_{k-1}-x_k)^2}{\tau} + \lambda \sigma^2 \tau \sum_{k=1}^{N} x_k^2
$$

Permanent impact often adds a term independent of schedule for complete liquidation, so the key schedule trade-off is temporary impact versus inventory risk.

Risk-neutral trader:

$$
\lambda=0
$$

trades evenly.

Risk-averse trader:

$$
\lambda>0
$$

front-loads execution.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class AlmgrenChrissConfig:
    initial_inventory: float = 1_000_000.0
    initial_price: float = 50.0
    horizon_days: float = 1.0
    n_intervals: int = 20
    daily_volatility: float = 0.025
    temporary_impact_eta: float = 2.5e-7
    permanent_impact_gamma: float = 1.0e-7
    risk_aversion: float = 1.0e-6
    spread_bps: float = 1.5
    commission_bps: float = 0.20
    adv_shares: float = 20_000_000.0
    max_participation: float = 0.15
    monte_carlo_paths: int = 20_000
    seed: int = 42
    output_subdir: str = "almgren_chriss_optimal_execution_v1"

config = AlmgrenChrissConfig()
config

## 6. Core schedule functions

We build:

1. TWAP schedule;
2. front-loaded exponential schedule;
3. back-loaded schedule;
4. Almgren-Chriss risk-averse schedule;
5. VWAP-style schedule using an intraday volume curve.

For the discrete Almgren-Chriss solution, the optimal inventory has hyperbolic-sine shape:

$$
x_k = X \frac{\sinh(\kappa(T-t_k))} {\sinh(\kappa T)}
$$

with:

$$
\kappa \approx \sqrt{ \frac{\lambda\sigma^2}{\eta} }
$$

This notebook uses a stable discrete approximation and then converts inventory path into trades.

In [ ]:
def time_grid(config):
    tau = config.horizon_days / config.n_intervals
    t = np.arange(config.n_intervals + 1) * tau
    return t, tau

def trades_from_inventory(inventory):
    inventory = np.asarray(inventory, dtype=float)
    return inventory[:-1] - inventory[1:]

def twap_inventory(config):
    return np.linspace(config.initial_inventory, 0.0, config.n_intervals + 1)

def exponential_inventory(config, speed=4.0):
    t, _ = time_grid(config)
    T = config.horizon_days
    raw = np.exp(-speed * t / T)
    inventory = config.initial_inventory * (raw - raw[-1]) / (raw[0] - raw[-1])
    inventory[0] = config.initial_inventory
    inventory[-1] = 0.0
    return inventory

def back_loaded_inventory(config, power=2.0):
    t, _ = time_grid(config)
    T = config.horizon_days
    remaining_fraction = 1.0 - (t / T) ** power
    inventory = config.initial_inventory * remaining_fraction
    inventory[0] = config.initial_inventory
    inventory[-1] = 0.0
    return inventory

def vwap_volume_curve(config, open_strength=0.35, close_strength=0.45):
    n = config.n_intervals
    u = np.linspace(0, 1, n)
    curve = (
        1.0
        + open_strength * np.exp(-((u - 0.05) / 0.18) ** 2)
        + close_strength * np.exp(-((u - 0.95) / 0.18) ** 2)
    )
    curve = curve / curve.sum()
    return curve

def vwap_inventory(config):
    curve = vwap_volume_curve(config)
    trades = config.initial_inventory * curve
    inventory = np.r_[config.initial_inventory, config.initial_inventory - np.cumsum(trades)]
    inventory[-1] = 0.0
    return inventory

def almgren_chriss_inventory(config, risk_aversion=None):
    if risk_aversion is None:
        risk_aversion = config.risk_aversion

    t, tau = time_grid(config)
    T = config.horizon_days

    if risk_aversion <= 0:
        return twap_inventory(config)

    sigma = config.daily_volatility
    eta = config.temporary_impact_eta

    kappa = np.sqrt(max(risk_aversion * sigma**2 / max(eta, 1e-18), 0.0))

    if kappa * T < 1e-6:
        return twap_inventory(config)

    numerator = np.sinh(kappa * (T - t))
    denominator = np.sinh(kappa * T)
    inventory = config.initial_inventory * numerator / denominator

    inventory[0] = config.initial_inventory
    inventory[-1] = 0.0
    return inventory

schedules = {
    "TWAP": twap_inventory(config),
    "VWAP": vwap_inventory(config),
    "FrontLoaded": exponential_inventory(config, speed=4.0),
    "BackLoaded": back_loaded_inventory(config, power=2.0),
    "AlmgrenChriss": almgren_chriss_inventory(config),
}

pd.DataFrame({name: inv for name, inv in schedules.items()}).head()

## 7. Visualise inventory and trading schedules

In [ ]:
t, tau = time_grid(config)

plt.figure(figsize=(12, 6))
for name, inventory in schedules.items():
    plt.plot(t, inventory / config.initial_inventory, marker="o", label=name)
plt.title("Inventory Remaining by Schedule")
plt.xlabel("Time")
plt.ylabel("Fraction of initial inventory")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
for name, inventory in schedules.items():
    trades = trades_from_inventory(inventory)
    plt.plot(np.arange(1, config.n_intervals + 1), trades / config.initial_inventory, marker="o", label=name)
plt.title("Trade List by Interval")
plt.xlabel("Interval")
plt.ylabel("Fraction of initial inventory traded")
plt.legend()
plt.show()

## 8. Deterministic expected cost and risk

For a schedule:

Temporary impact cost:

$$
C_{temp} = \sum_k \eta \frac{n_k^2}{\tau}
$$

Spread and commission cost:

$$
C_{linear} = \sum_k |n_k|S_0 \frac{spread/2+commission}{10000}
$$

Permanent impact cost approximation for full liquidation:

$$
C_{perm} \approx \frac{1}{2}\gamma X^2
$$

Inventory risk variance:

$$
Var(C) = \sigma^2 \tau S_0^2 \sum_k x_k^2
$$

The exact convention varies across implementations. The point is to separate expected impact from timing risk.

In [ ]:
def schedule_cost_risk(inventory, config):
    inventory = np.asarray(inventory, dtype=float)
    trades = trades_from_inventory(inventory)
    _, tau = time_grid(config)

    trade_rate = trades / tau

    temporary_cost = np.sum(config.temporary_impact_eta * trades**2 / tau)
    permanent_cost = 0.5 * config.permanent_impact_gamma * config.initial_inventory**2

    linear_bps = config.spread_bps / 2.0 + config.commission_bps
    spread_commission_cost = np.sum(np.abs(trades) * config.initial_price * linear_bps / 10000.0)

    inventory_risk_variance = (
        config.daily_volatility**2
        * tau
        * config.initial_price**2
        * np.sum(inventory[1:]**2)
    )
    inventory_risk_std = np.sqrt(inventory_risk_variance)

    expected_cost = temporary_cost + permanent_cost + spread_commission_cost
    objective = expected_cost + config.risk_aversion * inventory_risk_variance

    participation = np.abs(trades) / (config.adv_shares * tau / config.horizon_days)

    return {
        "temporary_cost": temporary_cost,
        "permanent_cost": permanent_cost,
        "spread_commission_cost": spread_commission_cost,
        "expected_cost": expected_cost,
        "inventory_risk_variance": inventory_risk_variance,
        "inventory_risk_std": inventory_risk_std,
        "objective": objective,
        "max_participation": participation.max(),
        "mean_participation": participation.mean(),
        "cost_bps_notional": expected_cost / (config.initial_inventory * config.initial_price) * 10000.0,
        "risk_bps_notional": inventory_risk_std / (config.initial_inventory * config.initial_price) * 10000.0,
    }

summary_rows = []
for name, inventory in schedules.items():
    row = {"schedule": name}
    row.update(schedule_cost_risk(inventory, config))
    summary_rows.append(row)

schedule_summary = pd.DataFrame(summary_rows).sort_values("objective")
schedule_summary

## 9. Expected cost versus risk chart

This is the execution efficient frontier idea:

- more urgent schedules usually cost more;
- slower schedules usually carry more risk;
- risk aversion chooses a point on the curve.

In [ ]:
plt.figure(figsize=(10, 6))
for _, row in schedule_summary.iterrows():
    plt.scatter(row["inventory_risk_std"], row["expected_cost"], s=80)
    plt.text(row["inventory_risk_std"], row["expected_cost"], row["schedule"])
plt.title("Expected Cost vs Inventory Risk")
plt.xlabel("Inventory risk standard deviation, dollars")
plt.ylabel("Expected execution cost, dollars")
plt.grid(True, alpha=0.3)
plt.show()

## 10. Almgren-Chriss efficient frontier over risk aversion

Vary $\lambda$.

Higher $\lambda$ means more urgency.

We compute:

- execution half-life;
- expected cost;
- risk;
- max participation;
- objective.

In [ ]:
lambda_grid = np.logspace(-10, -3, 30)

frontier_rows = []
frontier_inventories = {}

for lam in lambda_grid:
    cfg = AlmgrenChrissConfig(risk_aversion=float(lam))
    inv = almgren_chriss_inventory(cfg, risk_aversion=lam)
    metrics = schedule_cost_risk(inv, cfg)

    half_inventory = cfg.initial_inventory / 2
    below_half = np.where(inv <= half_inventory)[0]
    if len(below_half):
        half_life_time = below_half[0] * (cfg.horizon_days / cfg.n_intervals)
    else:
        half_life_time = np.nan

    row = {
        "risk_aversion": lam,
        "execution_half_life_days": half_life_time,
        **metrics,
    }
    frontier_rows.append(row)
    frontier_inventories[lam] = inv

frontier = pd.DataFrame(frontier_rows)

frontier.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(frontier["inventory_risk_std"], frontier["expected_cost"], marker="o")
plt.title("Almgren-Chriss Efficient Frontier")
plt.xlabel("Inventory risk standard deviation, dollars")
plt.ylabel("Expected execution cost, dollars")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 6))
plt.semilogx(frontier["risk_aversion"], frontier["execution_half_life_days"], marker="o")
plt.title("Execution Half-Life vs Risk Aversion")
plt.xlabel("Risk aversion lambda")
plt.ylabel("Execution half-life, days")
plt.grid(True, alpha=0.3)
plt.show()

## 11. Monte Carlo implementation shortfall

We simulate price paths:

$$
S_k = S_{k-1} + \sigma S_0 \sqrt{\tau}\epsilon_k - \gamma n_k
$$

For a sell order, execution price:

$$
\begin{aligned}
\tilde S_k &= S_{k-1} \\
&\quad - \eta\frac{n_k}{\tau} \\
&\quad - spreadCost \\
&\quad - commission
\end{aligned}
$$

Implementation shortfall:

$$
IS = X S_0 - \sum_k n_k \tilde S_k
$$

In [ ]:
def simulate_implementation_shortfall(inventory, config, n_paths=None, seed=None):
    if n_paths is None:
        n_paths = config.monte_carlo_paths
    if seed is None:
        seed = config.seed

    rng = np.random.default_rng(seed)
    trades = trades_from_inventory(inventory)
    _, tau = time_grid(config)
    n = len(trades)

    shocks = rng.standard_normal((n_paths, n))
    prices = np.full((n_paths, n + 1), config.initial_price, dtype=float)
    exec_prices = np.zeros((n_paths, n), dtype=float)

    linear_bps = config.spread_bps / 2.0 + config.commission_bps
    linear_cost_per_share = config.initial_price * linear_bps / 10000.0

    for k in range(n):
        pre_trade_price = prices[:, k]
        temporary_impact_per_share = config.temporary_impact_eta * trades[k] / tau

        exec_prices[:, k] = pre_trade_price - temporary_impact_per_share - linear_cost_per_share

        permanent_move = -config.permanent_impact_gamma * trades[k]
        random_move = config.daily_volatility * config.initial_price * np.sqrt(tau) * shocks[:, k]
        prices[:, k + 1] = pre_trade_price + permanent_move + random_move

    proceeds = exec_prices @ trades
    decision_value = config.initial_inventory * config.initial_price
    shortfall = decision_value - proceeds

    return pd.DataFrame({
        "implementation_shortfall": shortfall,
        "shortfall_bps": shortfall / decision_value * 10000.0,
        "terminal_price": prices[:, -1],
    })

mc_frames = []
for name, inventory in schedules.items():
    sim = simulate_implementation_shortfall(inventory, config, seed=config.seed + len(name))
    sim["schedule"] = name
    mc_frames.append(sim)

mc_results = pd.concat(mc_frames, ignore_index=True)

mc_results.head()

## 12. Monte Carlo shortfall summary

In [ ]:
mc_summary = (
    mc_results
    .groupby("schedule")
    .agg(
        mean_shortfall=("implementation_shortfall", "mean"),
        std_shortfall=("implementation_shortfall", "std"),
        p05_shortfall=("implementation_shortfall", lambda x: x.quantile(0.05)),
        p50_shortfall=("implementation_shortfall", lambda x: x.quantile(0.50)),
        p95_shortfall=("implementation_shortfall", lambda x: x.quantile(0.95)),
        mean_shortfall_bps=("shortfall_bps", "mean"),
        p95_shortfall_bps=("shortfall_bps", lambda x: x.quantile(0.95)),
    )
    .reset_index()
    .merge(schedule_summary, on="schedule", how="left")
    .sort_values("mean_shortfall")
)

mc_summary

In [ ]:
plt.figure(figsize=(12, 6))
for name, group in mc_results.groupby("schedule"):
    plt.hist(group["shortfall_bps"], bins=80, alpha=0.35, density=True, label=name)
plt.title("Implementation Shortfall Distribution")
plt.xlabel("Shortfall, bps of notional")
plt.ylabel("Density")
plt.legend()
plt.show()

## 13. Participation constraints

An execution plan may be mathematically optimal but operationally impossible.

Participation per interval:

$$
p_k = \frac{|n_k|}{ADV_k}
$$

If $p_k$ exceeds a maximum threshold, execution should be slowed, sliced differently, or rejected.

In [ ]:
def participation_report(inventory, config):
    trades = trades_from_inventory(inventory)
    curve = vwap_volume_curve(config)
    interval_adv = config.adv_shares * curve
    participation = np.abs(trades) / np.maximum(interval_adv, 1.0)

    return pd.DataFrame({
        "interval": np.arange(1, len(trades) + 1),
        "trade_shares": trades,
        "interval_adv_shares": interval_adv,
        "participation": participation,
        "breach_max_participation": participation > config.max_participation,
    })

participation_frames = []
for name, inventory in schedules.items():
    pr = participation_report(inventory, config)
    pr["schedule"] = name
    participation_frames.append(pr)

participation_table = pd.concat(participation_frames, ignore_index=True)

participation_summary = (
    participation_table
    .groupby("schedule")
    .agg(
        max_participation=("participation", "max"),
        mean_participation=("participation", "mean"),
        n_breach_intervals=("breach_max_participation", "sum"),
    )
    .reset_index()
    .sort_values("max_participation", ascending=False)
)

participation_summary

In [ ]:
plt.figure(figsize=(12, 6))
for name, group in participation_table.groupby("schedule"):
    plt.plot(group["interval"], group["participation"], marker="o", label=name)
plt.axhline(config.max_participation, linestyle="--", label="max participation")
plt.title("Participation by Interval")
plt.xlabel("Interval")
plt.ylabel("Participation")
plt.legend()
plt.show()

## 14. Sensitivity to volatility and temporary impact

Execution policy should be stress-tested.

Higher volatility increases risk of waiting.

Higher temporary impact discourages urgency.

In [ ]:
vol_grid = [0.010, 0.015, 0.025, 0.040, 0.060]
eta_grid = [1e-7, 2.5e-7, 5e-7, 1e-6]

sensitivity_rows = []

for vol in vol_grid:
    for eta in eta_grid:
        cfg = AlmgrenChrissConfig(
            daily_volatility=vol,
            temporary_impact_eta=eta,
            risk_aversion=config.risk_aversion,
            permanent_impact_gamma=config.permanent_impact_gamma,
            n_intervals=config.n_intervals,
            initial_inventory=config.initial_inventory,
            initial_price=config.initial_price,
            horizon_days=config.horizon_days,
            adv_shares=config.adv_shares,
            max_participation=config.max_participation,
        )
        inv = almgren_chriss_inventory(cfg)
        metrics = schedule_cost_risk(inv, cfg)
        sensitivity_rows.append({
            "daily_volatility": vol,
            "temporary_impact_eta": eta,
            **metrics,
        })

sensitivity = pd.DataFrame(sensitivity_rows)

sensitivity.head()

In [ ]:
pivot = sensitivity.pivot(index="daily_volatility", columns="temporary_impact_eta", values="cost_bps_notional")

plt.figure(figsize=(8, 6))
plt.imshow(pivot.values, aspect="auto")
plt.colorbar(label="Expected cost, bps")
plt.xticks(range(len(pivot.columns)), [f"{c:.1e}" for c in pivot.columns])
plt.yticks(range(len(pivot.index)), [f"{i:.3f}" for i in pivot.index])
plt.xlabel("Temporary impact eta")
plt.ylabel("Daily volatility")
plt.title("Expected Cost Sensitivity")
plt.show()

## 15. Calibration notes

The Almgren-Chriss model needs calibrated inputs:

| Parameter | Meaning | Possible calibration source |
|---|---|---|
| $\sigma$ | volatility | realised intraday volatility |
| $\eta$ | temporary impact | implementation shortfall vs participation |
| $\gamma$ | permanent impact | post-trade drift / metaorder studies |
| spread | bid/ask cost | L1 top-of-book data |
| ADV | liquidity | historical volume |
| $\lambda$ | risk aversion | desk mandate / utility calibration |

For Chinese futures or any futures market, additional calibration includes:

- tick size;
- contract multiplier;
- margin;
- price limits;
- night session liquidity;
- exchange fees;
- contract roll liquidity;
- L1 bid/ask availability.

## 16. Governance flags

Execution plans should be reviewed if:

1. participation breaches threshold;
2. expected cost is too high;
3. shortfall tail risk is too high;
4. strategy is overly sensitive to impact assumptions;
5. risk aversion produces overly aggressive front-loading;
6. input parameters are uncalibrated.

In [ ]:
selected_schedule = "AlmgrenChriss"
selected_summary = schedule_summary[schedule_summary["schedule"] == selected_schedule].iloc[0]
selected_mc = mc_summary[mc_summary["schedule"] == selected_schedule].iloc[0]
selected_participation = participation_summary[participation_summary["schedule"] == selected_schedule].iloc[0]

cost_limit_bps = 25.0
shortfall_p95_limit_bps = 75.0

sensitivity_cost_range = (
    sensitivity["cost_bps_notional"].max()
    - sensitivity["cost_bps_notional"].min()
)

governance_flags = pd.DataFrame([{
    "schedule": selected_schedule,
    "expected_cost_bps": selected_summary["cost_bps_notional"],
    "risk_bps": selected_summary["risk_bps_notional"],
    "p95_shortfall_bps": selected_mc["p95_shortfall_bps"],
    "max_participation": selected_participation["max_participation"],
    "n_participation_breaches": selected_participation["n_breach_intervals"],
    "sensitivity_cost_range_bps": sensitivity_cost_range,
    "flag_expected_cost_above_limit": bool(selected_summary["cost_bps_notional"] > cost_limit_bps),
    "flag_p95_shortfall_above_limit": bool(selected_mc["p95_shortfall_bps"] > shortfall_p95_limit_bps),
    "flag_participation_breach": bool(selected_participation["n_breach_intervals"] > 0),
    "flag_high_sensitivity_to_inputs": bool(sensitivity_cost_range > 50.0),
    "flag_uncalibrated_parameters": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_expected_cost_above_limit",
        "flag_p95_shortfall_above_limit",
        "flag_participation_breach",
        "flag_high_sensitivity_to_inputs",
        "flag_uncalibrated_parameters",
    ]
].any(axis=1)

governance_flags

## 17. Audit checks

Numerical checks:

1. inventory starts at $X$;
2. inventory ends at zero;
3. trades sum to $X$;
4. costs are non-negative;
5. Monte Carlo shortfalls are finite;
6. participation is finite;
7. frontier is finite.

In [ ]:
audit_rows = []

for name, inventory in schedules.items():
    trades = trades_from_inventory(inventory)

    audit_rows.append({
        "check": f"{name}_starts_at_initial_inventory",
        "value": float(inventory[0]),
        "passed": bool(abs(inventory[0] - config.initial_inventory) < 1e-8),
    })

    audit_rows.append({
        "check": f"{name}_ends_at_zero",
        "value": float(inventory[-1]),
        "passed": bool(abs(inventory[-1]) < 1e-8),
    })

    audit_rows.append({
        "check": f"{name}_trades_sum_to_initial_inventory",
        "value": float(trades.sum()),
        "passed": bool(abs(trades.sum() - config.initial_inventory) < 1e-6),
    })

costs_nonnegative = bool((schedule_summary[["temporary_cost", "permanent_cost", "spread_commission_cost", "expected_cost"]] >= -1e-12).all().all())
audit_rows.append({
    "check": "costs_nonnegative",
    "value": costs_nonnegative,
    "passed": costs_nonnegative,
})

mc_finite = bool(np.isfinite(mc_results[["implementation_shortfall", "shortfall_bps", "terminal_price"]].to_numpy()).all())
audit_rows.append({
    "check": "monte_carlo_outputs_finite",
    "value": mc_finite,
    "passed": mc_finite,
})

participation_finite = bool(np.isfinite(participation_table["participation"]).all())
audit_rows.append({
    "check": "participation_finite",
    "value": participation_finite,
    "passed": participation_finite,
})

frontier_finite = bool(np.isfinite(frontier.select_dtypes(include=[float, int]).to_numpy()).all())
audit_rows.append({
    "check": "frontier_numeric_outputs_finite",
    "value": frontier_finite,
    "passed": frontier_finite,
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 18. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

# Save schedule paths.
schedule_path = output_dir / "execution_schedules.csv"
schedule_table = []
for name, inventory in schedules.items():
    trades = trades_from_inventory(inventory)
    for k in range(config.n_intervals):
        schedule_table.append({
            "schedule": name,
            "interval": k + 1,
            "time_start": t[k],
            "time_end": t[k + 1],
            "inventory_start": inventory[k],
            "inventory_end": inventory[k + 1],
            "trade_shares": trades[k],
            "trade_fraction_initial": trades[k] / config.initial_inventory,
        })

schedule_table = pd.DataFrame(schedule_table)
schedule_table.to_csv(schedule_path, index=False)

schedule_summary.to_csv(output_dir / "schedule_summary.csv", index=False)
frontier.to_csv(output_dir / "efficient_frontier.csv", index=False)
mc_results.to_csv(output_dir / "monte_carlo_shortfall_paths.csv", index=False)
mc_summary.to_csv(output_dir / "monte_carlo_shortfall_summary.csv", index=False)
participation_table.to_csv(output_dir / "participation_table.csv", index=False)
participation_summary.to_csv(output_dir / "participation_summary.csv", index=False)
sensitivity.to_csv(output_dir / "vol_impact_sensitivity.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "almgren_chriss_optimal_execution_outputs",
    "schema_version": "almgren_chriss_optimal_execution_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "schedules": list(schedules.keys()),
    "selected_schedule": selected_schedule,
    "selected_schedule_summary": selected_summary.to_dict(),
    "selected_monte_carlo_summary": selected_mc.to_dict(),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook implements a simplified Almgren-Chriss optimal execution model.",
        "Temporary impact, permanent impact, spread, commission and inventory risk are separated.",
        "Risk aversion controls the shape of the optimal execution schedule.",
        "TWAP, VWAP, front-loaded, back-loaded and Almgren-Chriss schedules are compared.",
        "Monte Carlo simulation estimates implementation shortfall distributions.",
        "Participation constraints and parameter sensitivity are reported.",
        "Parameters are illustrative and should be calibrated from execution data before production use."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

schedule_path, output_dir / "schedule_summary.csv", output_dir / "monte_carlo_shortfall_summary.csv", output_dir / "governance_flags.csv", manifest_path

## 19. Practical checklist for optimal execution

Before using an execution schedule:

1. **Calibrate volatility** using intraday data.
2. **Calibrate spread** from L1 bid/ask.
3. **Calibrate temporary impact** from fills or historical metaorders.
4. **Calibrate permanent impact** carefully; it is hard to estimate.
5. **Check participation** against ADV and interval volume.
6. **Stress volatility and impact assumptions.**
7. **Compare to TWAP and VWAP baselines.**
8. **Report implementation shortfall distribution.**
9. **Define risk aversion through desk mandate.**
10. **Add production risk controls before trading.**

## 20. Limitations

### Linear impact assumptions

The model uses simplified linear temporary and permanent impact.

Real impact is nonlinear, transient, regime-dependent and venue-dependent.

### No order book

The notebook does not model queue position, order book depth, hidden liquidity or limit order behaviour.

### No intraday volume uncertainty

VWAP curve is deterministic.

Real intraday volume is stochastic.

### No price limits or halts

Important for futures and some equity markets.

### No adaptive execution

The schedule is precomputed and does not react to realised price, volume or fills.

### Calibration is illustrative

Parameters are synthetic. Production execution requires historical fill and market data calibration.

## 21. Research frontier and extensions

Important extensions include:

1. nonlinear impact;
2. transient impact;
3. stochastic liquidity;
4. adaptive execution;
5. VWAP/POV algorithms;
6. limit-order placement;
7. queue position;
8. multi-asset portfolio execution;
9. execution with alpha signals;
10. futures execution with tick size, margin, night sessions and price limits.

## 22. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_02_limit_order_book_simulator**  
   Move from schedule-level execution to order-book mechanics.

2. **06_03_market_impact_model_calibration**  
   Estimate impact parameters from simulated or real fills.

3. **06_04_execution_algorithms_twap_vwap_pov**  
   Convert execution schedules into algorithmic execution styles.

4. **06_07_latency_and_queue_position_model**  
   Model queue risk and latency.

5. **06_11_l1_bid_ask_backtest_execution_model**  
   Use L1 bid/ask data for realistic backtest fills.

## 23. Summary

This notebook implemented the Almgren-Chriss optimal execution model.

It showed how to:

1. define implementation shortfall;
2. model temporary impact;
3. model permanent impact;
4. include inventory risk;
5. derive risk-neutral and risk-averse intuition;
6. generate TWAP, VWAP, front-loaded, back-loaded and AC schedules;
7. compute expected cost and inventory risk;
8. build an execution efficient frontier;
9. simulate implementation shortfall with Monte Carlo;
10. analyse participation constraints;
11. stress volatility and impact parameters;
12. create governance flags and audit checks;
13. save outputs and manifests.

The key computational takeaway:

> Almgren-Chriss converts execution into a controlled optimisation problem over inventory trajectories.

The key financial takeaway:

> Trading faster reduces timing risk but increases impact; trading slower reduces impact but leaves inventory exposed.

## 24. Further reading

- Almgren and Chriss, "Optimal Execution of Portfolio Transactions."
- Almgren, "Optimal Trading with Stochastic Liquidity and Volatility."
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Gatheral and Schied on market impact and optimal execution.
- Bouchaud, Farmer and Lillo on market impact.